# Dataset Stats Check
Quick check of the key numbers after pulling fresh data from MongoDB:
- Unique videos found
- Unique categories found
- Usable videos (has both early-window AND target-window snapshot)

Run `pull_from_mongodb.py` first to generate the input file this reads from.

In [1]:
import pandas as pd

RAW_PATH ="C:/Users/Lenovo/youtube_enagagement_project/data/raw/youtube_mongodb_export.csv"

EARLY_CUTOFF = 6            # hours
TARGET_LOW, TARGET_HIGH = 18, 30   # hours

df = pd.read_csv(RAW_PATH)
df = df.dropna(subset=["video_id", "video_age_hours"])

print("Total rows:", df.shape[0])
print("Unique videos found:", df["video_id"].nunique())
print("Unique categories found:", df["category_id"].nunique())
print("Category list:", sorted(df["category_id"].unique()))

Total rows: 105718
Unique videos found: 654
Unique categories found: 14
Category list: [np.float64(1.0), np.float64(2.0), np.float64(10.0), np.float64(15.0), np.float64(17.0), np.float64(19.0), np.float64(20.0), np.float64(22.0), np.float64(23.0), np.float64(24.0), np.float64(25.0), np.float64(26.0), np.float64(27.0), np.float64(28.0)]


## Videos per category
Helps spot which categories are still thin on data.

In [2]:
print("Videos per category:")
print(df.groupby("category_id")["video_id"].nunique().sort_values(ascending=False))

Videos per category:
category_id
15.0    266
22.0    145
24.0     58
20.0     56
27.0     31
10.0     23
25.0     15
26.0     13
1.0      13
23.0      9
2.0       8
17.0      7
28.0      7
19.0      3
Name: video_id, dtype: int64


## Early-window / target-window / usable video counts

In [3]:
has_early = set(df[df["video_age_hours"] <= EARLY_CUTOFF]["video_id"].unique())
has_target = set(df[(df["video_age_hours"] >= TARGET_LOW) &
                     (df["video_age_hours"] <= TARGET_HIGH)]["video_id"].unique())
usable_videos = has_early & has_target

print(f"Videos with early snapshot (<= {EARLY_CUTOFF}h): {len(has_early)}")
print(f"Videos with target snapshot ({TARGET_LOW}-{TARGET_HIGH}h): {len(has_target)}")
print(f"Usable videos (both): {len(usable_videos)}")

Videos with early snapshot (<= 6h): 621
Videos with target snapshot (18-30h): 569
Usable videos (both): 537


## Usable videos per category
This is your real new training sample size, broken down by category.

In [4]:
usable_df = df[df["video_id"].isin(usable_videos)]
print("Usable videos per category:")
print(usable_df.groupby("category_id")["video_id"].nunique().sort_values(ascending=False))

Usable videos per category:
category_id
15.0    225
22.0    116
24.0     48
20.0     43
27.0     23
10.0     21
25.0     13
26.0     11
1.0      10
17.0      7
23.0      7
2.0       5
28.0      5
19.0      3
Name: video_id, dtype: int64
